In [ ]:
import io
import re
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# CONFIGURATION
# ==========================================

# GCS path to evaluation you want to compare
PATH_TSV_A = "gs://fc-secure-8c7c6bb6-6241-4a03-b87d-895b5abfd91d/submissions/final-outputs/4615fe9e-4d23-4902-9654-9b0b577e5289/VcfdistEvaluation/d34a1cd7-863a-435d-88b8-f423b85bab3b/call-SummarizeEvaluations/evaluation_summary.tsv"

# GCS path pointing to chr20 of Sam's original run
PATH_TSV_B = "gs://fc-secure-e9018a40-98de-4d5d-8f40-16e60c8f8a0b/submissions/249dea13-d27c-405a-8158-c174f81f85ef/VcfdistEvaluation/78f80114-6660-43ed-8293-57084d33c929/call-SummarizeEvaluations/evaluation_summary.tsv"

LABEL_RUN_A = "reshaped panel chr20"
LABEL_RUN_B = "Sam's original chr20"

OUTPUT_PREFIX = "vcfdist_comparison"


# Fixed y-axis range for difference plots so separate notebook runs are directly comparable.
# Set to None to fall back to autoscaling from data.
FIXED_DIFF_Y_RANGE = (-0.1, 0.1)


In [ ]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================

def parse_gcs_url(gcs_url: str):
    """Split a gs://bucket/prefix URL into (bucket_name, prefix)."""
    match = re.match(r"gs://([^/]+)/(.*)", gcs_url)
    if not match:
        raise ValueError(f"Invalid GCS URL: {gcs_url}")
    bucket_name, prefix = match.groups()
    return bucket_name, prefix.rstrip("/")


def read_tsv(path: str, storage_client=None) -> pd.DataFrame:
    """Read an evaluation_summary TSV from a local path or a GCS path (gs://).

    Parameters
    ----------
    path:
        Local filesystem path or a GCS URL (``gs://bucket/...``).
    storage_client:
        An already-authenticated :class:`google.cloud.storage.Client`.
        Required only when *path* starts with ``gs://``.
        If ``None`` and a GCS path is provided, a default client will be
        created automatically.

    Returns
    -------
    pd.DataFrame
        Indexed by the region-name column (first column of the TSV).
        Column names are the metric names (e.g. ``SNP_F1_SCORE``).
    """
    if path.startswith("gs://"):
        from google.cloud import storage as gcs

        client = storage_client or gcs.Client()
        bucket_name, blob_path = parse_gcs_url(path)
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_path)
        raw = blob.download_as_bytes()
        return pd.read_csv(io.BytesIO(raw), sep="\t", index_col=0)
    else:
        return pd.read_csv(path, sep="\t", index_col=0)


In [ ]:
# ==========================================
# LOAD DATA
# ==========================================

# If reading from GCS, a storage client is created lazily inside read_tsv().
# You may optionally create one explicitly and pass it:
#   from google.cloud import storage
#   _client = storage.Client()
#   df_a = read_tsv(PATH_TSV_A, storage_client=_client)
#   df_b = read_tsv(PATH_TSV_B, storage_client=_client)

df_a = read_tsv(PATH_TSV_A)
df_b = read_tsv(PATH_TSV_B)

print(f"Loaded Run A ({LABEL_RUN_A}): {df_a.shape[0]} regions, {df_a.shape[1]} metrics")
print(f"Loaded Run B ({LABEL_RUN_B}): {df_b.shape[0]} regions, {df_b.shape[1]} metrics")


In [ ]:
# ==========================================
# INSPECT RAW DATA
# ==========================================

print(f"=== Run A: {LABEL_RUN_A} ===")
display(df_a)

print(f"\n=== Run B: {LABEL_RUN_B} ===")
display(df_b)


In [ ]:
# ==========================================
# COMPUTE DIFFERENCES  (Run A − Run B)
# ==========================================

# Align on common regions
common_regions = df_a.index.intersection(df_b.index)
df_diff = df_a.loc[common_regions].copy() - df_b.loc[common_regions].copy()

# Variant types and their core metrics
VARIANT_TYPES = ["SNP", "INDEL", "SV"]
METRICS        = ["PREC", "RECALL", "F1_SCORE"]

print("Differences (Run A − Run B) for F1, Precision, Recall:")
cols_to_show = [f"{v}_{m}" for v in VARIANT_TYPES for m in METRICS]
display(df_diff[cols_to_show].round(4))


In [ ]:
# ==========================================
# PLOT: COMPARISON  (grouped bar charts)
# ==========================================

def plot_metric_comparison(df_a, df_b, label_a, label_b,
                            variant_type, metric,
                            ax, region_short_names=None):
    """Side-by-side bar chart for one (variant_type, metric) combination."""
    col = f"{variant_type}_{metric}"
    regions = df_a.index.intersection(df_b.index)
    vals_a = df_a.loc[regions, col].values
    vals_b = df_b.loc[regions, col].values

    x = np.arange(len(regions))
    width = 0.38

    bars_a = ax.bar(x - width / 2, vals_a, width, label=label_a, color="steelblue", alpha=0.85)
    bars_b = ax.bar(x + width / 2, vals_b, width, label=label_b, color="coral",     alpha=0.85)

    ax.set_title(f"{variant_type} — {metric}", fontsize=10)
    ax.set_xticks(x)
    short = region_short_names if region_short_names else list(regions)
    ax.set_xticklabels(short, rotation=35, ha="right", fontsize=7)
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_tick_params(labelsize=7)
    ax.legend(fontsize=6)
    ax.axhline(1.0, color="gray", linewidth=0.6, linestyle="--")


# Short labels for regions (drop long boilerplate prefixes)
REGION_SHORT = {
    "notinalldifficultregions":               "notinDifficult",
    "notinAllTandemRepeatsandHomopolymers":    "notInTR/HP",
    "AllTandemRepeatsandHomopolymers":         "AllTR/HP",
    "AllTandemRepeats":                        "AllTR",
    "AllHomopolymers_ge7bp_imperfectge11bp":   "AllHP≥7bp",
    "All":                                     "All",
}

regions = df_a.index.intersection(df_b.index)
short_names = [REGION_SHORT.get(r, r) for r in regions]

n_variants = len(VARIANT_TYPES)
n_metrics  = len(METRICS)
fig, axes  = plt.subplots(n_variants, n_metrics,
                           figsize=(5 * n_metrics, 4 * n_variants),
                           constrained_layout=True)

for row, vtype in enumerate(VARIANT_TYPES):
    for col, metric in enumerate(METRICS):
        plot_metric_comparison(
            df_a, df_b, LABEL_RUN_A, LABEL_RUN_B,
            vtype, metric,
            axes[row][col],
            region_short_names=short_names,
        )

fig.suptitle(f"VcfdistEvaluation Comparison\n{LABEL_RUN_A}  vs  {LABEL_RUN_B}",
             fontsize=13, y=1.01)

plt.savefig(f"{OUTPUT_PREFIX}.comparison.png", bbox_inches="tight", dpi=150)
plt.savefig(f"{OUTPUT_PREFIX}.comparison.pdf", bbox_inches="tight")
plt.show()
print(f"Saved comparison plots to {OUTPUT_PREFIX}.comparison.[png|pdf]")


In [ ]:
# ==========================================
# PLOT: DIFFERENCES  (Run A - Run B)
# ==========================================

def plot_metric_diff(df_diff, variant_type, metric, ax,
                     y_limits=None, region_short_names=None):
    """Bar chart of (Run A - Run B) for one (variant_type, metric) combination."""
    col = f"{variant_type}_{metric}"
    regions = df_diff.index
    diffs = df_diff[col].values

    x = np.arange(len(regions))
    colors = ["steelblue" if v >= 0 else "coral" for v in diffs]

    ax.bar(x, diffs, color=colors, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"{variant_type} - {metric}", fontsize=10)
    ax.set_xticks(x)
    short = region_short_names if region_short_names else list(regions)
    ax.set_xticklabels(short, rotation=35, ha="right", fontsize=7)
    ax.yaxis.set_tick_params(labelsize=7)
    if y_limits is not None:
        ax.set_ylim(y_limits)


regions = df_diff.index
short_names = [REGION_SHORT.get(r, r) for r in regions]

# Use a fixed y-axis range across notebook runs for direct comparability.
if FIXED_DIFF_Y_RANGE is None:
    diff_cols = [f"{v}_{m}" for v in VARIANT_TYPES for m in METRICS]
    all_diffs = df_diff[diff_cols].to_numpy(dtype=float).ravel()
    all_diffs = all_diffs[np.isfinite(all_diffs)]
    max_abs_diff = np.max(np.abs(all_diffs)) if all_diffs.size else 0.01
    max_abs_diff = max(max_abs_diff, 0.01)
    y_limits = (-max_abs_diff, max_abs_diff)
else:
    y_limits = FIXED_DIFF_Y_RANGE

fig, axes = plt.subplots(n_variants, n_metrics,
                          figsize=(5 * n_metrics, 3.5 * n_variants),
                          constrained_layout=True)

for row, vtype in enumerate(VARIANT_TYPES):
    for col, metric in enumerate(METRICS):
        plot_metric_diff(df_diff, vtype, metric,
                         axes[row][col],
                         y_limits=y_limits,
                         region_short_names=short_names)
        if col == 0:
            axes[row][col].set_ylabel("delta (A - B)", fontsize=9)

fig.suptitle(
    f"VcfdistEvaluation Differences\n{LABEL_RUN_A} - {LABEL_RUN_B}\n"
    "Blue = A > B  |  Orange = A < B",
    fontsize=12, y=1.01,
)

plt.savefig(f"{OUTPUT_PREFIX}.differences.png", bbox_inches="tight", dpi=150)
plt.savefig(f"{OUTPUT_PREFIX}.differences.pdf", bbox_inches="tight")
plt.show()
print(f"Saved difference plots to {OUTPUT_PREFIX}.differences.[png|pdf]")


In [ ]:
# ==========================================
# SUMMARY TABLE
# ==========================================

cols_f1 = [f"{v}_F1_SCORE" for v in VARIANT_TYPES]
regions  = df_a.index.intersection(df_b.index)

summary_rows = []
for region in regions:
    for vtype in VARIANT_TYPES:
        col = f"{vtype}_F1_SCORE"
        val_a = df_a.loc[region, col]
        val_b = df_b.loc[region, col]
        delta = val_a - val_b
        summary_rows.append({
            "Region":        REGION_SHORT.get(region, region),
            "VariantType":   vtype,
            f"F1 {LABEL_RUN_A}": round(val_a, 4),
            f"F1 {LABEL_RUN_B}": round(val_b, 4),
            "Δ F1 (A−B)":   round(delta, 4),
        })

summary_df = pd.DataFrame(summary_rows)
print("F1 Score Summary:")
display(summary_df.style.background_gradient(
    subset=["Δ F1 (A−B)"], cmap="RdYlGn", vmin=-0.05, vmax=0.05
))
